# LSA Dataset Explorer

Exploración end-to-end para validación de Fase 0 del proyecto Avatar AI.

**Preguntas a responder:**
- ¿Los videos tienen subtítulos? ¿Son usables?
- ¿Están sincronizados?
- ¿MediaPipe extrae keypoints de buena calidad?
- ¿Qué estructura de dataset tiene sentido para Juan?

In [ ]:
import sys
sys.path.insert(0, '../scripts')

import json
from pathlib import Path
import yaml
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
import mediapipe as mp
import numpy as np

with open('../config.yaml') as f:
    config = yaml.safe_load(f)

print('Config cargada OK')

## Sección 1 — Descarga de video GCBA

Pegar una URL de YouTube de un video GCBA con intérprete LSA.

In [ ]:
from download import download_video

# Cambiar por URL real de video GCBA
VIDEO_URL = "https://www.youtube.com/watch?v=REEMPLAZAR"

download_video(VIDEO_URL, config)

# Listar archivos descargados
raw_files = list(Path(config['paths']['raw']).glob('*'))
sub_files = list(Path(config['paths']['subtitles']).glob('*'))
print('Videos:', [f.name for f in raw_files])
print('Subtítulos:', [f.name for f in sub_files])

In [ ]:
# Seleccionar video y subtítulo para el análisis
VIDEO_PATH = sorted(Path(config['paths']['raw']).glob('*.mp4'))[0]
SUB_PATH = sorted(Path(config['paths']['subtitles']).glob('*.srt'))[0]

print(f'Video: {VIDEO_PATH.name}')
print(f'Subtítulo: {SUB_PATH.name}')

# Metadatos del video
cap = cv2.VideoCapture(str(VIDEO_PATH))
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
duration = total_frames / fps
cap.release()

print(f'\nResolución: {width}x{height}')
print(f'FPS: {fps:.1f}')
print(f'Duración: {duration:.1f}s ({duration/60:.1f} min)')
print(f'Frames totales: {total_frames}')

## Sección 2 — Análisis de subtítulos

In [ ]:
from extract_subs import load_subtitles

segments = load_subtitles(SUB_PATH)
print(f'Total segmentos: {len(segments)}')
print()

df = pd.DataFrame([
    {'start': s.start, 'end': s.end, 'duration': s.end - s.start, 'text': s.text}
    for s in segments
])
print(df.head(10).to_string(index=False))

In [ ]:
# Métricas de riqueza
all_text = ' '.join(df['text'])
words = all_text.lower().split()
unique_words = set(words)

print(f'Palabras totales: {len(words)}')
print(f'Vocabulario único: {len(unique_words)}')
print(f'Cobertura temporal: {df["duration"].sum():.1f}s de {duration:.1f}s ({df["duration"].sum()/duration*100:.1f}%)')
print(f'Duración promedio por segmento: {df["duration"].mean():.2f}s')

## Sección 3 — Chequeo de sincronización

In [ ]:
from sync_subs import analyze_sync

sync = analyze_sync(VIDEO_PATH, SUB_PATH)
print('Resultado de sincronización:')
for k, v in sync.items():
    status = '✓' if k == 'sync_ok' and v else ('✗' if k == 'sync_ok' else ' ')
    print(f'  {status} {k}: {v}')

In [ ]:
# Visualizar timeline de subtítulos sobre duración del video
fig, ax = plt.subplots(figsize=(14, 2))
ax.barh(0, duration, color='lightgray', height=0.4, label='Video')
for _, row in df.iterrows():
    ax.barh(0, row['duration'], left=row['start'], color='steelblue', height=0.4, alpha=0.7)

ax.set_xlabel('Tiempo (segundos)')
ax.set_yticks([])
ax.set_title(f'Cobertura de subtítulos — {sync["coverage_ratio"]*100:.1f}% del video')
patches = [mpatches.Patch(color='lightgray', label='Video'), mpatches.Patch(color='steelblue', alpha=0.7, label='Subtítulos')]
ax.legend(handles=patches, loc='upper right')
plt.tight_layout()
plt.show()

## Sección 4 — Extracción de keypoints con MediaPipe

In [ ]:
from extract_keypoints import extract_keypoints

# Procesar primer segmento como prueba
seg = segments[0]
frame_start = int(seg.start * fps)
frame_end = int(seg.end * fps)

print(f'Procesando segmento 0: "{seg.text}"')
print(f'Frames {frame_start} → {frame_end} ({frame_end - frame_start} frames)')

kp_result = extract_keypoints(VIDEO_PATH, config, frame_start, frame_end)
print(f'\nFrames procesados: {kp_result["n_frames_processed"]}')
print(f'Confidence promedio: {kp_result["confidence_avg"]}')
print(f'Tiene pose en frame 0: {kp_result["frames"][0]["pose"] is not None}')
print(f'Tiene mano derecha en frame 0: {kp_result["frames"][0]["right_hand"] is not None}')
print(f'Tiene mano izquierda en frame 0: {kp_result["frames"][0]["left_hand"] is not None}')

In [ ]:
# Visualizar overlay de keypoints sobre un frame
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils

cap = cv2.VideoCapture(str(VIDEO_PATH))
cap.set(cv2.CAP_PROP_POS_FRAMES, frame_start)
ret, frame = cap.read()
cap.release()

rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
with mp_holistic.Holistic(
    model_complexity=config['mediapipe']['model_complexity'],
    min_detection_confidence=config['mediapipe']['min_detection_confidence'],
    static_image_mode=True
) as holistic:
    results = holistic.process(rgb)

annotated = rgb.copy()
mp_drawing.draw_landmarks(annotated, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS)
mp_drawing.draw_landmarks(annotated, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
mp_drawing.draw_landmarks(annotated, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)

plt.figure(figsize=(10, 6))
plt.imshow(annotated)
plt.title(f'MediaPipe Holistic — frame {frame_start}\n"{seg.text}"')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Analizar confianza a lo largo de todos los frames del segmento
confidences = [
    np.mean([lm['visibility'] for lm in f['pose']]) if f['pose'] else 0
    for f in kp_result['frames']
]
frame_nums = [f['frame'] for f in kp_result['frames']]

plt.figure(figsize=(12, 3))
plt.plot(frame_nums, confidences, color='steelblue')
plt.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='threshold 0.5')
plt.xlabel('Frame')
plt.ylabel('Confidence promedio (pose)')
plt.title(f'Calidad de detección MediaPipe — segmento 0')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Frames con confidence > 0.5: {sum(c > 0.5 for c in confidences)}/{len(confidences)}')

## Sección 5 — Armado de dataset mínimo

In [ ]:
# Procesar los primeros N segmentos para generar dataset de muestra
N_SEGMENTS = 5  # ajustar según tiempo disponible

dataset = []
for seg in segments[:N_SEGMENTS]:
    f_start = int(seg.start * fps)
    f_end = int(seg.end * fps)
    kp = extract_keypoints(VIDEO_PATH, config, f_start, f_end)

    entry = {
        "id": f"{VIDEO_PATH.stem}_{seg.index:04d}",
        "gloss": seg.text,
        "source": VIDEO_PATH.stem,
        "frames": {"start": f_start, "end": f_end},
        "keypoints": kp["frames"],
        "metadata": {
            "intent": None,          # para dataset propio: ej. 'renovar_dni'
            "tramite": None,
            "subtitle_type": "auto_cc",
            "sync_ok": sync["sync_ok"],
            "confidence_avg": kp["confidence_avg"],
            "fps": fps,
            "time_start": seg.start,
            "time_end": seg.end,
        }
    }
    dataset.append(entry)
    print(f'  [{seg.index}] "{seg.text[:50]}" — confidence: {kp["confidence_avg"]}')

print(f'\nDataset: {len(dataset)} entradas')

In [ ]:
# Guardar dataset
output_path = Path(config['paths']['dataset']) / f"{VIDEO_PATH.stem}_sample.json"
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(dataset, f, indent=2, ensure_ascii=False)

print(f'Dataset guardado en: {output_path}')

# Preview de estructura
entry = dataset[0]
preview = {k: v for k, v in entry.items() if k != 'keypoints'}
preview['keypoints'] = f'[{len(entry["keypoints"])} frames, cada uno con pose/left_hand/right_hand]'
print('\nEstructura de una entrada:')
print(json.dumps(preview, indent=2, ensure_ascii=False))

## Resumen — GO / NO-GO

In [ ]:
avg_confidence = np.mean([e['metadata']['confidence_avg'] for e in dataset])

print('=' * 50)
print('RESUMEN FASE 0')
print('=' * 50)
print(f'Video: {VIDEO_PATH.name}')
print(f'Duración: {duration:.0f}s')
print(f'Subtítulos encontrados: {"SÍ" if len(segments) > 0 else "NO"} ({len(segments)} segmentos)')
print(f'Sincronización: {"OK" if sync["sync_ok"] else "DESAJUSTADA"}')
print(f'Cobertura subtítulos: {sync["coverage_ratio"]*100:.1f}%')
print(f'Keypoints MediaPipe: confidence promedio = {avg_confidence:.3f}')
print(f'Dataset de muestra: {len(dataset)} entradas guardadas')
print()
go = len(segments) > 0 and sync['sync_ok'] and avg_confidence > 0.5
print(f'DECISIÓN: {"✓ GO — material usable" if go else "✗ NO-GO — revisar fuente"}')
print('=' * 50)